# microsimflow on Google Colab: Interactive Recipe Builder

This notebook provides a robust Graphical User Interface (GUI) that wraps the `microsimflow` CLI pipeline. It adopts an architecture completely separated from the backend, avoiding the reimplementation of the simulation's core logic within the GUI itself. When generating previews, it calls `run_pipeline.py` in the background with the `--solver skip` mode and directly renders the generated `.raw` binaries.

**Hardware Requirement:**
The GUI itself runs smoothly on a standard CPU runtime. A **T4 GPU** is required *only* if you intend to execute the `chfem` solver for actual property calculations in Step 4. (To change, go to `Runtime` > `Change runtime type` > Select `T4 GPU`).

**Main Features & Key Capabilities:**
* **Environment Agnostic:** Works seamlessly on both Google Colab and local Jupyter environments.
* **100% Backend Compatibility:** Since it directly calls the backend via subprocess, orientation parameters (`kappa`, `mean_dir`) and any newly added filler features can be used and previewed without requiring UI code updates.
* **Deformation Optimization:** The step editor automatically groups steps that share the same recipe. This allows heavy Random Sequential Adsorption (RSA) placements to run only once, efficiently applying bulk affine transformations (Stretch Ratios) afterward.
* **Automatic Default Recipe Entry:** When changing the filler type in the Sweep Generator, the corresponding default recipe string (including orientation parameters) is automatically populated.
* **Standardized 3D Preview:** The output coordinate system has been remapped to render in a standard isometric view (↑Z ←Y →X) with the Z-axis pointing upwards.
* **CSV Import/Export:** You can load past experiment logs (`.csv`) to restore step lists for re-editing or previewing, and export optimized batch `.py` scripts.

### Step 1: Setup Workspace & Install Dependencies
We clone the `microsimflow` repository, configure the workspace, and set the `PYTHONPATH`. We also install `plotly`, `scikit-image`, and `pyvista` for robust 3D rendering in Colab.

In [ ]:
import os
import sys
import subprocess

# 1. Environment check (Colab or Local)
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Google Colab environment detected. Cloning the repository...")
    if not os.path.exists('/content/microsimflow'):
        subprocess.run(['git', 'clone', 'https://github.com/hikuram/microsimflow.git', '/content/microsimflow'])
    os.environ["PYTHONPATH"] = "/content/microsimflow:" + os.environ.get("PYTHONPATH", "")
    subprocess.run(['pip', 'install', 'plotly', 'scikit-image', 'numba', 'pyvista', '-q'])
else:
    print("Local environment detected. Using the current directory as the workspace.")
    os.environ["PYTHONPATH"] = os.getcwd() + ":" + os.environ.get("PYTHONPATH", "")


### Step 2: Initialize the GUI Application
Run this cell to define the `CLIWrapperApp`. This interface handles global settings, filler recipes, 3D previews, and Python script generation.

In [ ]:
import datetime
import shutil
import itertools
import textwrap
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
from skimage.measure import marching_cubes

class CLIWrapperApp:
    def __init__(self):
        # Build session-based cache directory
        self.session_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.cache_dir = os.path.join(".gui_preview_cache", self.session_id)
        os.makedirs(self.cache_dir, exist_ok=True)
        
        self.output_log = widgets.Output()
        self.build_ui()
        self.update_slider_range() # Initialize

    def build_ui(self):
        # --- Global Settings ---
        self.grid_size = widgets.IntText(value=100, description='Grid Size:', style={'description_width': 'initial'})
        self.voxel_size = widgets.FloatText(value=1e-8, description='Voxel(m):', style={'description_width': 'initial'})
        self.bg_type = widgets.Dropdown(
            options=['single', 'gyroid', 'lamellar', 'cylinder', 'bcc', 'sea_island', 'island_sea'],
            value='single', description='BG Type:', style={'description_width': 'initial'}
        )
        self.physics_mode = widgets.Dropdown(
            options=['thermal', 'electrical', 'mechanics', 'permeability'],
            value='electrical', description='Physics:', style={'description_width': 'initial'}
        )
        self.solver = widgets.Dropdown(
            options=['chfem', 'puma', 'both', 'skip'],
            value='chfem', description='Solver:', style={'description_width': 'initial'}
        )
        global_settings = widgets.VBox([
            widgets.HTML("<b>Global Pipeline Settings</b>"),
            widgets.HBox([self.grid_size, self.voxel_size, self.bg_type]),
            widgets.HBox([self.physics_mode, self.solver])
        ])

        # --- Tab 1: Sweep Generator ---
        self.sweep_container = widgets.VBox([self.create_sweep_row()])
        
        btn_add_sweep = widgets.Button(description="Add Filler Range", icon="plus")
        btn_add_sweep.on_click(self.add_sweep_row)
        
        self.stretch_min = widgets.FloatText(value=1.0, step=0.1, description='Stretch Min:', style={'description_width': 'initial'})
        self.stretch_max = widgets.FloatText(value=1.0, step=0.1, description='Stretch Max:', style={'description_width': 'initial'})
        self.stretch_steps = widgets.IntText(value=1, description='Steps:', style={'description_width': 'initial'})
        stretch_ui = widgets.HBox([self.stretch_min, self.stretch_max, self.stretch_steps])

        btn_generate_steps = widgets.Button(description="Generate Steps ->", button_style="info")
        btn_generate_steps.on_click(self.generate_steps_from_sweep)
        
        self.tab1 = widgets.VBox([
            widgets.HTML("<b>Sweep Generator:</b> Specify parameter ranges to generate steps in bulk."),
            self.sweep_container, 
            btn_add_sweep, 
            widgets.HTML("<hr><b>Deformation (Stretch Ratio)</b>"), 
            stretch_ui,
            widgets.HTML("<hr>"),
            btn_generate_steps
        ])

        # --- Tab 2: Step Editor & CSV Import ---
        self.step_list_vbox = widgets.VBox([])
        
        self.csv_path_input = widgets.Text(value='comparison_results.csv', description='CSV Path:', style={'description_width': 'initial'})
        self.csv_readonly_check = widgets.Checkbox(value=False, description='Lock steps (Read-only)')
        btn_import_csv = widgets.Button(description="Import CSV", button_style="warning")
        btn_import_csv.on_click(self.import_csv)
        
        btn_add_manual_step = widgets.Button(description="Add Manual Step", icon="plus")
        btn_add_manual_step.on_click(lambda _: self.add_step_row("", 1.0, False))

        btn_clear_steps = widgets.Button(description="Clear All Steps", button_style="danger")
        btn_clear_steps.on_click(self.clear_all_steps)

        self.tab2 = widgets.VBox([
            widgets.HBox([self.csv_path_input, self.csv_readonly_check, btn_import_csv]),
            widgets.HTML("<hr><b>Execution Steps (CLI Arguments)</b>"),
            self.step_list_vbox,
            widgets.HTML("<hr>"),
            widgets.HBox([btn_add_manual_step, btn_clear_steps])
        ])

        # --- Tab 3: Run, Preview & Export ---
        # Batch Run Area
        btn_run_selected = widgets.Button(description="Run Selected Recipe", button_style="success", layout={'width': '220px'})
        btn_run_selected.on_click(self.run_selected_recipe)
        
        btn_run_all = widgets.Button(description="Run All Recipes", button_style="warning", layout={'width': '220px'})
        btn_run_all.on_click(self.run_all_recipes)
        
        self.run_log_textarea = widgets.Textarea(
            value="Ready.\n", 
            layout={'width': '100%', 'height': '150px'},
            disabled=True
        )

        # Preview Control Area
        self.preview_slider = widgets.IntSlider(min=0, max=0, description='Target Step:', layout={'width': '400px'}, style={'description_width': 'initial'})
        self.preview_slider.observe(self.on_preview_slider_change, names='value')
        
        self.step_info_html = widgets.HTML(value="<div style='padding:10px; background:#f0f8ff; border-radius:5px;'><i>Select a step to preview.</i></div>")

        preview_warning_html = widgets.HTML(
            "<small style='color: #666;'>"
            "Note: The 3D preview is cropped to max 100³ voxels for performance. "
            "Due to Plotly (go.Mesh3d) WebGL transparency limitations, overlapping fillers may exhibit rendering artifacts; "
            "use ParaView with exported .vtkhdf for accurate visualization."
            "</small>"
        )

        self.preview_out = widgets.Output(layout={'border': '1px solid #ddd', 'min_height': '500px', 'padding': '10px'})
        with self.preview_out:
            print("3D Preview will appear here after generation...")

        # Export Script Area
        self.script_name_input = widgets.Text(value=f'run_batch_{self.session_id}.py', description='Export Name:', style={'description_width': 'initial'})
        btn_export_script = widgets.Button(description="Export Batch Script", button_style="primary")
        btn_export_script.on_click(self.export_batch_script)

        self.tab3 = widgets.VBox([
            widgets.HTML("<b>1. Cache Generation</b><br><small>* Steps with the same recipe are automatically grouped. Placement recalculation is skipped for efficient bulk deformation.</small>"),
            widgets.HBox([btn_run_selected, btn_run_all]),
            self.run_log_textarea,
            widgets.HTML("<hr><b>2. Interactive Preview</b>"),
            self.preview_slider,
            self.step_info_html,
            preview_warning_html,
            self.preview_out,
            widgets.HTML("<hr><b>3. Script Export</b>"),
            widgets.HBox([self.script_name_input, btn_export_script])
        ])

        # --- Tab Layout Assembly ---
        self.tab_selector = widgets.ToggleButtons(
            options=['1. Sweep Generator', '2. Step List Editor', '3. Preview & Export'],
            style={'button_width': '180px'}
        )
        self.tab_content_area = widgets.VBox([self.tab1], layout={'border': '1px solid #ccc', 'padding': '15px', 'margin': '10px 0'})

        def on_tab_change(change):
            if change['new'] == '1. Sweep Generator':
                self.tab_content_area.children = [self.tab1]
            elif change['new'] == '2. Step List Editor':
                self.tab_content_area.children = [self.tab2]
            else:
                self.tab_content_area.children = [self.tab3]
                
        self.tab_selector.observe(on_tab_change, names='value')

        self.main_ui = widgets.VBox([
            widgets.HTML(f"<h2>microsimflow GUI - Session: {self.session_id}</h2>"),
            global_settings, 
            widgets.HTML("<hr>"),
            self.tab_selector,
            self.tab_content_area, 
            self.output_log
        ])

    # --- Core Logic: Recipe Grouping (Optimization) ---
    def _get_recipe_groups(self):
        """Automatically group steps with the same recipe string into a single Basename and multiple Stretches"""
        groups = []
        recipe_to_group = {}
        for idx, row in enumerate(self.step_list_vbox.children):
            recipe_str = row.children[1].value
            stretch_val = row.children[3].value
            
            if recipe_str not in recipe_to_group:
                g_idx = len(groups)
                new_group = {
                    'group_idx': g_idx,
                    'basename': os.path.join(self.cache_dir, f"recipe_group_{g_idx:03d}"),
                    'recipe': recipe_str,
                    'stretches': [],
                    'step_indices': []
                }
                groups.append(new_group)
                recipe_to_group[recipe_str] = new_group
            
            group = recipe_to_group[recipe_str]
            if stretch_val not in group['stretches']:
                group['stretches'].append(stretch_val)
            group['step_indices'].append(idx)
            
        return groups

    # --- State Management Logic ---
    def update_slider_range(self):
        max_val = max(0, len(self.step_list_vbox.children) - 1)
        self.preview_slider.max = max_val
        self.on_preview_slider_change(None)

    # --- Tab 1 Logic ---
    def create_sweep_row(self):
        # Default parameters mapped to filler types
        default_params = {
            'rigidfiber': "length=60:radius=2:mean_dir=0,0,1:kappa=0.0",
            'flake': "radius=10:thickness=2:mean_dir=0,0,1:kappa=0.0",
            'sphere': "radius=5",
            'irregfiber': "length=60:shape=bean:radius=5:ratio=0.5:mean_dir=0,0,1:kappa=0.0",
            'flexfiber': "length=90:radius=2:max_bend_deg=90:max_total_bends=10",
            'agglomerate': "num_fibers=5:length=90:radius=2:max_bend_deg=90",
            'staggered': "radius=15:layer_thickness=2:min_layers=1:max_layers=4:max_offset_pct=30:mean_dir=0,0,1:kappa=0.0"
        }
        
        f_type = widgets.Dropdown(options=list(default_params.keys()))
        vf_min = widgets.FloatText(value=0.05, step=0.01, description='Vf Min:', style={'description_width': 'initial'})
        vf_max = widgets.FloatText(value=0.05, step=0.01, description='Vf Max:', style={'description_width': 'initial'})
        vf_steps = widgets.IntText(value=1, description='Steps:', style={'description_width': 'initial'})
        params = widgets.Text(value=default_params['rigidfiber'], description='Params:', style={'description_width': 'initial'})
        
        # Callback to auto-update default parameters when dropdown changes
        def on_type_change(change):
            if change['type'] == 'change' and change['name'] == 'value':
                params.value = default_params.get(change['new'], "")
                
        f_type.observe(on_type_change)
        
        btn_del = widgets.Button(icon="trash", layout={'width': '40px'}, button_style="danger")
        row = widgets.HBox([f_type, vf_min, vf_max, vf_steps, params, btn_del])
        
        # Safely rebuild tuple inside callback to avoid consecutive DOM approaches
        def on_del(_):
            children = list(self.sweep_container.children)
            if row in children:
                children.remove(row)
                self.sweep_container.children = tuple(children)
        btn_del.on_click(on_del)
        return row

    def add_sweep_row(self, _):
        self.sweep_container.children = tuple(list(self.sweep_container.children) + [self.create_sweep_row()])

    def generate_steps_from_sweep(self, _):
        vf_lists = []
        fillers_meta = []
        for row in self.sweep_container.children:
            ftype = row.children[0].value
            vmin = row.children[1].value
            vmax = row.children[2].value
            vsteps = max(1, row.children[3].value)
            params = row.children[4].value
            
            vfs = np.linspace(vmin, vmax, vsteps).tolist() if vsteps > 1 else [vmin]
            vf_lists.append(vfs)
            fillers_meta.append((ftype, params))

        s_steps = max(1, self.stretch_steps.value)
        stretch_ratios = np.linspace(self.stretch_min.value, self.stretch_max.value, s_steps).tolist() if s_steps > 1 else [self.stretch_min.value]
        combinations = list(itertools.product(*vf_lists))
        
        # Generate in bulk and assign as tuple (minimize frontend communication)
        new_rows = []
        for stretch in stretch_ratios:
            for combo in combinations:
                recipe_parts = []
                for idx, vf in enumerate(combo):
                    if vf > 0:
                        ftype, params = fillers_meta[idx]
                        recipe_parts.append(f"{ftype}:{vf:.4f}:{params}")
                
                if recipe_parts:
                    recipe_str = " ".join(recipe_parts)
                    new_rows.append(self.create_step_row(recipe_str, stretch, False))
                    
        self.step_list_vbox.children = tuple(new_rows)
        self.update_slider_range()
        
        with self.output_log:
            clear_output(wait=True)
            print(f"✅ Generated {len(self.step_list_vbox.children)} steps from sweep settings.")

    # --- Tab 2 Logic ---
    def create_step_row(self, recipe_str, stretch_val, is_readonly):
        txt_recipe = widgets.Text(value=recipe_str, disabled=is_readonly)
        txt_stretch = widgets.FloatText(value=stretch_val, disabled=is_readonly)
        btn_del = widgets.Button(icon="trash", layout={'width': '40px'}, button_style="danger", disabled=is_readonly)
        row = widgets.HBox([
            widgets.Label("Recipe:"), txt_recipe, 
            widgets.Label("Stretch:"), txt_stretch, 
            btn_del
        ])
        
        def on_del(_):
            children = list(self.step_list_vbox.children)
            if row in children:
                children.remove(row)
                self.step_list_vbox.children = tuple(children)
                self.update_slider_range()
        btn_del.on_click(on_del)
        return row

    def add_step_row(self, recipe_str, stretch_val, is_readonly):
        self.step_list_vbox.children = tuple(list(self.step_list_vbox.children) + [self.create_step_row(recipe_str, stretch_val, is_readonly)])
        self.update_slider_range()

    def clear_all_steps(self, _):
        self.step_list_vbox.children = ()
        self.update_slider_range()

    def import_csv(self, _):
        path = self.csv_path_input.value
        if not os.path.exists(path):
            with self.output_log: 
                clear_output(wait=True)
                print(f"❌ CSV not found: {path}")
            return
        try:
            df = pd.read_csv(path)
            is_readonly = self.csv_readonly_check.value
            new_rows = []
            for _, row in df.iterrows():
                recipe = str(row.get('Recipe', 'none:0.0'))
                stretch = float(row.get('Stretch_Ratio', 1.0))
                new_rows.append(self.create_step_row(recipe, stretch, is_readonly))
            self.step_list_vbox.children = tuple(new_rows)
            self.update_slider_range()
            with self.output_log: 
                clear_output(wait=True)
                print(f"✅ Successfully imported {len(new_rows)} steps from {path}")
        except Exception as e:
            with self.output_log: 
                clear_output(wait=True)
                print(f"❌ Failed to parse CSV: {e}")

    # --- Tab 3 Logic (Batch Run & Interactive Preview) ---
    def _execute_group(self, group):
        self.run_log_textarea.value += f"--- Group {group['group_idx']} ---\n"
        self.run_log_textarea.value += f"Recipe: {group['recipe']}\n"
        self.run_log_textarea.value += f"Stretches: {group['stretches']}\n"
        
        real_size = self.grid_size.value
        
        stretch_args = []
        for s in group['stretches']:
            stretch_args.append(str(s))
            
        cmd = [
            "python3", "-m", "run_pipeline",
            "--size", str(real_size),
            "--voxel_size", str(self.voxel_size.value),
            "--bg_type", self.bg_type.value,
            "--physics_mode", self.physics_mode.value,
            "--solver", "skip", 
            "--basename", group['basename'],
            "--stretch_ratios"
        ] + stretch_args + ["--recipe"] + group['recipe'].split()

        try:
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                self.run_log_textarea.value += line
            p.wait()
            if p.returncode != 0:
                self.run_log_textarea.value += f"❌ Error occurred in Group {group['group_idx']}.\n"
        except Exception as e:
            self.run_log_textarea.value += f"❌ Exception: {e}\n"

    def run_selected_recipe(self, _):
        idx = self.preview_slider.value
        groups = self._get_recipe_groups()
        
        target_group = None
        for g in groups:
            if idx in g['step_indices']:
                target_group = g
                break
                
        if not target_group:
            self.run_log_textarea.value = "❌ Cannot find recipe group for the selected step.\n"
            return
            
        self.run_log_textarea.value = f"🚀 Starting execution for Selected Recipe (Batching {len(target_group['stretches'])} deformations)...\n\n"
        self._execute_group(target_group)
        self.run_log_textarea.value += "\n✅ Execution completed.\n"
        self.on_preview_slider_change(None) # Re-render

    def run_all_recipes(self, _):
        groups = self._get_recipe_groups()
        if not groups:
            self.run_log_textarea.value = "❌ No steps available to run.\n"
            return

        self.run_log_textarea.value = f"🚀 Starting batch execution for {len(groups)} unique recipes...\n\n"
        for g in groups:
            self._execute_group(g)
            
        self.run_log_textarea.value += "\n✅ Batch generation completed.\n"
        self.on_preview_slider_change(None) # Re-render

    def on_preview_slider_change(self, change):
        idx = self.preview_slider.value
        rows = self.step_list_vbox.children
        if idx < 0 or idx >= len(rows):
            self.step_info_html.value = "<div style='padding:10px; background:#fce4e4; border-radius:5px;'><i>No steps available. Please add steps in Tab 1 or 2.</i></div>"
            with self.preview_out:
                clear_output()
            return
            
        target_row = rows[idx]
        recipe_str = target_row.children[1].value
        stretch_val = target_row.children[3].value

        # Immediate HTML feedback for selection state
        self.step_info_html.value = f"<div style='padding:10px; background:#eef6fc; border-left:4px solid #3b82f6; border-radius:3px;'><b>Target Step:</b> {idx} &nbsp;&nbsp;|&nbsp;&nbsp; <b>Stretch Ratio:</b> {stretch_val}<br><b>Recipe:</b> <code>{recipe_str}</code></div>"
        
        # Render immediately if cache exists
        self.render_cached_preview(idx, stretch_val)

    def render_cached_preview(self, idx, stretch_val):
        groups = self._get_recipe_groups()
        target_group = None
        for g in groups:
            if idx in g['step_indices']:
                target_group = g
                break
                
        if not target_group:
            return

        real_size = self.grid_size.value
        poisson_ratio = 0.4
        raw_file = f"{target_group['basename']}_L{stretch_val:.2f}.raw"

        with self.preview_out:
            clear_output(wait=True)
            if not os.path.exists(raw_file):
                print(f"⚠️ Cache not found for Step {idx} (Expected: {os.path.basename(raw_file)}).\nPlease click 'Run Selected Recipe' or 'Run All Recipes' to generate the structure.")
                return
                
            print(f"Loading cache: {os.path.basename(raw_file)} ...")
            
            lam = stretch_val
            lam_nu = stretch_val ** (-poisson_ratio)
            
            # Calculate the correct full grid size and restore the binary
            nz = max(1, int(round(real_size * lam_nu)))
            ny = max(1, int(round(real_size * lam_nu)))
            nx = max(1, int(round(real_size * lam)))
            
            try:
                final_grid = np.fromfile(raw_file, dtype=np.uint8).reshape((nz, ny, nx))
            except ValueError as e:
                print(f"❌ Failed to reshape binary file. Dimension mismatch: {e}")
                return
                
            # Crop (slice) to a maximum of 100^3 for preview rendering
            pz, py, px = min(nz, 100), min(ny, 100), min(nx, 100)
            preview_grid = final_grid[:pz, :py, :px]

            unique_ids = np.unique(preview_grid)
            filler_ids = unique_ids[unique_ids >= 4]
            
            traces = []
            colors = ['coral', 'royalblue', 'mediumseagreen', 'gold', 'orchid', 'cyan']
            for i, f_id in enumerate(filler_ids):
                mask = (preview_grid == f_id).astype(float)
                if np.any(mask):
                    verts, faces, _, _ = marching_cubes(mask, level=0.5)
                    # Fix: Map (Z,Y,X) input to X=verts[:,2], Y=verts[:,1], Z=verts[:,0] for Plotly
                    # Swapping mappings so Z is upwards (z), Y is left (y), and X is right (x) in 3D space.
                    traces.append(go.Mesh3d(
                        x=verts[:, 2], y=verts[:, 1], z=verts[:, 0], 
                        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], 
                        color=colors[i % len(colors)], name=f"ID {f_id}", opacity=0.8
                    ))

            if not traces:
                print("ℹ️ No fillers found in the cropped preview area.")
                return

            fig = go.Figure(data=traces)
            fig.update_layout(
                scene=dict(
                    aspectmode='data',
                    xaxis=dict(range=[0, px], title='X'),
                    yaxis=dict(range=[0, py], title='Y'),
                    zaxis=dict(range=[0, pz], title='Z'),
                    # Adjust camera view: Isometric view (Z up, from front-right)
                    camera=dict(
                        up=dict(x=0, y=0, z=1),
                        eye=dict(x=1.5, y=-1.5, z=1.2)
                    )
                ),
                margin=dict(l=0, r=0, b=0, t=30),
                title=f"3D Preview: Step {idx} (Cropped Size={px}x{py}x{pz} / Original={nx}x{ny}x{nz})",
                height=500
            )
            display(fig)

    def export_batch_script(self, _):
        groups = self._get_recipe_groups()
        if not groups:
            with self.output_log: 
                clear_output(wait=True)
                print("❌ No steps to export.")
            return
            
        out_filename = self.script_name_input.value
        
        # Build the output script using the same highly efficient grouping design
        tasks_code = []
        for g in groups:
            tasks_code.append(f"        {{'recipe': '{g['recipe']}', 'stretches': {g['stretches']}}},")
        tasks_str = "[\n" + "\n".join(tasks_code) + "\n    ]"

        template = textwrap.dedent(f"""\
import os
import subprocess

def run_batch():
    out_dir = "result_{self.session_id}"
    os.makedirs(out_dir, exist_ok=True)
    csv_log = f"{{out_dir}}_results.csv"
    
    print("--- Starting GUI Explicit Batch Sweep (Optimized) ---")
    tasks = {tasks_str}

    for idx, task in enumerate(tasks):
        basename = os.path.join(out_dir, f"recipe_group_{{idx:03d}}")
        recipe_args = task['recipe'].split()
        stretch_args = [str(s) for s in task['stretches']]
        
        cmd = [
            "python3", "-m", "run_pipeline",
            "--size", "{self.grid_size.value}",
            "--voxel_size", "{self.voxel_size.value}",
            "--bg_type", "{self.bg_type.value}",
            "--physics_mode", "{self.physics_mode.value}",
            "--solver", "{self.solver.value}",
            "--basename", basename, 
            "--csv_log", csv_log, 
            "--stretch_ratios"
        ] + stretch_args + ["--recipe"] + recipe_args

        print(f"\\nRunning Recipe Group {{idx+1}}/{{len(tasks)}} (Stretches: {{task['stretches']}})...")
        subprocess.run(cmd)

if __name__ == "__main__":
    run_batch()
""")
        try:
            with open(out_filename, 'w') as f:
                f.write(template)
            with self.output_log: 
                clear_output(wait=True)
                print(f"✅ Successfully exported optimized batch script '{out_filename}'.")
        except Exception as e:
            with self.output_log: 
                clear_output(wait=True)
                print(f"❌ Export Error: {e}")


### Step 3: Launch the GUI
Run the cell below to launch the GUI. All operations are isolated into a session-specific cache directory (`.gui_preview_cache/YYYYMMDD_HHMMSS/`).

In [ ]:
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

app = CLIWrapperApp()
display(app.main_ui)


### Step 4: Install Additional Dependencies & Compile `chfem` Solver (GPU Required)
**Note:** Run this cell *only* if you selected `chfem` as your solver in the GUI and wish to execute the calculation suite now. This step requires a **T4 GPU** runtime.

In [ ]:
# 1. Verify CUDA compiler
!nvcc --version

# 2. Clone and compile chfem (using the fork with the typo fix)
!git clone https://github.com/hikuram/chfem.git /content/chfem
!cd /content/chfem && mkdir -p build && cd build && cmake .. \
      -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_CUDA_ARCHITECTURES=75 \
      -DCMAKE_C_FLAGS="-DCUDAPCG_MATKEY_32BIT" \
      -DCMAKE_CUDA_FLAGS="-DCUDAPCG_MATKEY_32BIT" && \
    make -j2

# 3. Install Python dependencies
!pip install pyvista numba

import os
# Add compiled chfem to system PATH
os.environ['PATH'] = '/content/chfem:/content/microsimflow:' + os.environ['PATH']

### Step 5: Execute the Simulation
After configuring your structure and generating the script in Step 3, run the cell below to execute the batch calculation in your workspace.

In [ ]:
#!python3 run_batch_YYYYMMDD_HHMMSS.py